# Week 6 -- Function 4: GP + UCB Exploitation (4D)
**Warehouse Product Placement** — maximise Y toward zero (currently beating baseline by less than it should)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 30
DATA_PATH_X  = '../data/function_4/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_4/initial_outputs.npy'

# W5 BREAKTHROUGH: -8.93 vs previous best -24.5
# Pattern: lower values across all dims = better; corners are catastrophic
weekly_log = [
    ([0.888889, 0.555556, 0.777778, 0.777778], -24.548023981997797),  # W1: -24.5
    ([1.000000, 0.666667, 0.111111, 0.777778], -28.563568040174733),  # W2: worse
    ([1.000000, 0.666667, 0.111111, 0.777778], -28.563568040174733),  # W3: EI stuck
    ([1.000000, 1.000000, 1.000000, 0.000000], -48.00036259295127),   # W4: corner disaster
    ([0.629449, 0.425195, 0.523474, 0.108441], -8.933668034974371),   # W5: BEST (-8.93)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (35, 4) | Y: (35,)
Week 6 | 35 total observations (30 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 4: Warehouse Placement (4D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending -- all negative, maximise means least negative is best
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.4f}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.4f}  (least negative)')
print(f'  Best X*  = [{best_x_str}]')

Input  shape : (35, 4)
Output shape : (35,)

  All observations (sorted descending by Y)
  [ 1]  X = [0.577766, 0.428772, 0.425826, 0.249007]   Y = -4.0255  <-- best
  [ 2]  X = [0.326076, 0.472367, 0.453192, 0.105887]   Y = -6.7021
  [ 3]  X = [0.282138, 0.505987, 0.530531, 0.096302]   Y = -7.9668
  [ 4]  X = [0.629449, 0.425195, 0.523474, 0.108441]   Y = -8.9337
  [ 5]  X = [0.124871, 0.129770, 0.384400, 0.287076]   Y = -10.0696
  [ 6]  X = [0.170347, 0.756959, 0.276520, 0.531231]   Y = -11.5657
  [ 7]  X = [0.250946, 0.033693, 0.145380, 0.494932]   Y = -11.6999
  [ 8]  X = [0.247708, 0.060445, 0.042186, 0.441324]   Y = -12.6817
  [ 9]  X = [0.626071, 0.586751, 0.438806, 0.778858]   Y = -12.7418
  [10]  X = [0.216911, 0.166086, 0.241372, 0.770062]   Y = -12.7583
  [11]  X = [0.738613, 0.482103, 0.709366, 0.503970]   Y = -13.1228
  [12]  X = [0.732812, 0.145250, 0.476813, 0.133366]   Y = -13.5276
  [13]  X = [0.037825, 0.664853, 0.161982, 0.253924]   Y = -13.7027
  [14]  X = [0.889356

In [3]:
# Cell 4: Fit GP
# F4 values range -8 to -48, a normal scale -- no log transform needed.
# length_scale=0.3 so the GP can extrapolate into the focused search region.
# alpha=1e-4 allows slight noise tolerance.
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.3, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

# W5 anchor point
w5_anchor = np.array([0.629449, 0.425195, 0.523474, 0.108441])

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Alpha (noise)           : 1e-4')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X*:')
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'    X*                     = [{best_x_str}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}')
print(f'    Actual Y*              = {best_Y:.4f}')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
pred_w5, std_w5 = gp.predict(w5_anchor.reshape(1, -1), return_std=True)
print(f'  GP at W5 anchor [{" ".join(f"{v:.3f}" for v in w5_anchor)}]:')
print(f'    predicted mean         = {pred_w5[0]:.4f}  (actual: -8.9337)')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.3)
  Alpha (noise)           : 1e-4
  Log-marginal-likelihood : -3831.4824
  Sanity check at best known X*:
    X*                     = [0.577766, 0.428772, 0.425826, 0.249007]
    GP predicted mean      = -4.0263
    Actual Y*              = -4.0255
    GP predicted std       = 0.00999675
  GP at W5 anchor [0.629 0.425 0.523 0.108]:
    predicted mean         = -8.9333  (actual: -8.9337)


In [4]:
# Cell 5: GP UCB Acquisition -- 4D grid around W5 breakthrough
# W5 [0.629, 0.425, 0.523, 0.108] scored -8.93. All evidence: lower = better.
# Push each dimension slightly lower while staying close to the breakthrough.

x1 = np.linspace(0.40, 0.75, 20)
x2 = np.linspace(0.25, 0.55, 20)
x3 = np.linspace(0.35, 0.65, 20)
x4 = np.linspace(0.00, 0.20, 20)
xx1, xx2, xx3, xx4 = np.meshgrid(x1, x2, x3, x4)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel()])

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

# beta=1.0: exploit the W5 breakthrough region
beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  GP UCB Acquisition (beta=1.0, exploitation)')
print('=' * 72)
print(f'  Grid : x1=[0.40,0.75] x x2=[0.25,0.55] x x3=[0.35,0.65] x x4=[0.00,0.20]')
print(f'         20^4 = 160,000 points')
print(f'  Beta : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  UCB score    : {ucb_scores[best_idx]:.4f}')
print(f'  GP mean      : {gp_mean[best_idx]:.4f}')
print(f'  GP std       : {gp_std[best_idx]:.6f}')
print('=' * 72)

  GP UCB Acquisition (beta=1.0, exploitation)
  Grid : x1=[0.40,0.75] x x2=[0.25,0.55] x x3=[0.35,0.65] x x4=[0.00,0.20]
         20^4 = 160,000 points
  Beta : 1.0  (exploitation)
  Next query   : [0.528947, 0.471053, 0.350000, 0.200000]
  UCB score    : -3.1883
  GP mean      : -3.3981
  GP std       : 0.209761


In [5]:
# Cell 6: Sanity check -- stay near the W5 breakthrough

w5_anchor = np.array([0.629449, 0.425195, 0.523474, 0.108441])
dist_to_w5   = np.linalg.norm(next_x - w5_anchor)
dist_to_best = np.linalg.norm(next_x - best_X)
in_zone = (
    0.40 <= next_x[0] <= 0.75 and
    0.25 <= next_x[1] <= 0.55 and
    0.35 <= next_x[2] <= 0.65 and
    0.00 <= next_x[3] <= 0.20
)

print('=' * 72)
print('  Sanity Check')
print('=' * 72)
next_str = ', '.join(f'{v:.6f}' for v in next_x)
w5_str   = ', '.join(f'{v:.6f}' for v in w5_anchor)
best_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Proposed query    : [{next_str}]')
print(f'  W5 anchor         : [{w5_str}]  (Y=-8.93, best weekly)')
print(f'  Best known X*     : [{best_str}]  (Y={best_Y:.4f})')
print(f'  Distance to W5    : {dist_to_w5:.6f}')
print(f'  Distance to X*    : {dist_to_best:.6f}')
print(f'  Inside target zone: {in_zone}')
print()

if dist_to_w5 > 0.40:
    print('  WARNING: query is far from W5 breakthrough (> 0.40). Risk of leaving good region!')
else:
    print('  OK: query is close to W5 breakthrough.')

if not in_zone:
    print('  WARNING: query is outside target zone. Corners caused W3/W4 disasters!')
else:
    print('  OK: query is inside the target zone.')
print('=' * 72)

  Sanity Check
  Proposed query    : [0.528947, 0.471053, 0.350000, 0.200000]
  W5 anchor         : [0.629449, 0.425195, 0.523474, 0.108441]  (Y=-8.93, best weekly)
  Best known X*     : [0.577766, 0.428772, 0.425826, 0.249007]  (Y=-4.0255)
  Distance to W5    : 0.225122
  Distance to X*    : 0.111005
  Inside target zone: True

  OK: query is close to W5 breakthrough.
  OK: query is inside the target zone.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 4 -- Warehouse Placement (4D)')
print('  Objective       : Maximise Y (all negative, want least negative)')
print(f'  Method          : GP UCB (beta=1.0, 4D grid, raw Y fit, length_scale=0.3)')
print(f'  W5 breakthrough : -8.93 at [0.629, 0.425, 0.523, 0.108] -- exploit this region')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.4f}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 4 -- Warehouse Placement (4D)
  Objective       : Maximise Y (all negative, want least negative)
  Method          : GP UCB (beta=1.0, 4D grid, raw Y fit, length_scale=0.3)
  W5 breakthrough : -8.93 at [0.629, 0.425, 0.523, 0.108] -- exploit this region

  Best X*         : [0.577766, 0.428772, 0.425826, 0.249007]
  Best Y*         : -4.0255

  Next query      : [0.528947, 0.471053, 0.350000, 0.200000]

  Portal submission string:
  >>> 0.528947-0.471053-0.350000-0.200000 <<<


## Week 6 -- Reflection

**Function**: F4 -- Warehouse Product Placement (4D)

### Strategy change from Week 5
- Removed NN: sent W4 to corner [1,1,1,0] → -48, worst result ever.
- Removed SVM: couldn't distinguish good from bad with this data.
- Raw Y fit, length_scale=0.3: GP can now extrapolate across the focused grid.
- 4D grid centred on W5 breakthrough [0.629, 0.425, 0.523, 0.108].
- All grid bounds shifted lower: evidence says lower values = better Y.

### W5 breakthrough pattern
| Week | Y | Notes |
|------|---|-------|
| W1 | -24.5 | high dims |
| W4 | -48.0 | corner disaster |
| W5 | -8.93 | lower dims -- best ever |

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*